# Separate validation — unseen datasets
Evaluate frozen DRS-only LOSO checkpoints on two separate validation datasets: four unseen scattering-coefficient sets and three unseen white-matter sets. Each new condition is compared with the same held-out subject's RMSE on the original validation dataset.

## Parameters
Set `TARGET` to `'hc'` or `'sto2'`. The notebook preserves the LOSO pairing and applies each checkpoint's training-fold normalization to both new datasets.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():  # VS Code may start in notebooks/
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from final_refactored or its notebooks folder.')
os.chdir(ROOT)

SRC_DIR = ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + SUBPROCESS_ENV.get('PYTHONPATH', '')

TARGET = 'sto2'  # 'hc' or 'sto2'
FOLDS = None     # None for all folds, or e.g. '1,3,10-20'
DEVICE = 'cuda'  # 'auto', 'cpu', 'cuda', or e.g. 'cuda:1'
BATCH_SIZE = 4096
ROWS_PER_CONDITION = 12960
SAVE_PREDICTIONS = False

if TARGET not in {'hc', 'sto2'}:
    raise ValueError("TARGET must be 'hc' or 'sto2'")
CHECKPOINT_DIR = Path(f'artifacts/stage2/baseline/{TARGET}')
ORIGINAL_RESULTS = CHECKPOINT_DIR / 'loso_results.csv'
ARTIFACT_ROOT = Path('artifacts/stage2/unseen_dataset_validation')
FIGURE_DIR = Path('results/figures/stage2/unseen_dataset_validation')
UNIT = 'µM' if TARGET == 'hc' else 'fraction'
DATASETS = {
    'unseen_scattering': {
        'label': 'Unseen scattering coefficients',
        'mat': Path('data/NIRS_Absolute_Val_Dataset_new.mat'),
        'conditions': 4,
    },
    'unseen_wm': {
        'label': 'Unseen white-matter conditions',
        'mat': Path('data/NIRS_Absolute_Val_Dataset_diffwm.mat'),
        'conditions': 3,
    },
}

TARGET, CHECKPOINT_DIR, ORIGINAL_RESULTS, DATASETS

## 1. Evaluate both unseen datasets
Each fold evaluates only its LOSO test subject. Dataset-specific fold JSON files make inference resumable.

In [ ]:
for dataset_key, specification in DATASETS.items():
    validation_mat = specification['mat']
    output_dir = ARTIFACT_ROOT / dataset_key / TARGET
    if not validation_mat.is_file():
        raise FileNotFoundError(f'Missing validation dataset: {validation_mat}')
    command = [
        sys.executable, '-m', 'subject_nirs.stage2.unseen_scattering',
        '--target', TARGET,
        '--checkpoint_dir', str(CHECKPOINT_DIR),
        '--validation_mat', str(validation_mat),
        '--output_dir', str(output_dir),
        '--device', DEVICE,
        '--batch_size', str(BATCH_SIZE),
        '--rows_per_sim', str(ROWS_PER_CONDITION),
    ]
    if FOLDS:
        command.extend(['--folds', FOLDS])
    if not SAVE_PREDICTIONS:
        command.append('--no_save_predictions')
    print(f"\n=== {specification['label']} ===")
    subprocess.run(command, cwd=ROOT, env=SUBPROCESS_ENV, check=True)

## 2. Plot paired subject-level ΔRMSE
For subject/fold $i$ and unseen condition $k$, $\Delta RMSE_{ik}=RMSE_{ik}^{new}-RMSE_i^{original}$. Positive values indicate worse performance. The panel annotations report the median change, percentage of subjects that worsened, and Holm-adjusted paired Wilcoxon p-value.

In [ ]:
if not ORIGINAL_RESULTS.is_file():
    raise FileNotFoundError(f'Missing original LOSO results: {ORIGINAL_RESULTS}')
plot_command = [
    sys.executable, '-m', 'subject_nirs.stage2.unseen_scattering_plot',
    '--original_csv', str(ORIGINAL_RESULTS),
    '--target', TARGET,
    '--output_dir', str(FIGURE_DIR),
    '--output_prefix', f'fig_unseen_dataset_validation_{TARGET}_delta_rmse',
    '--unit', UNIT,
    '--title', f'{TARGET.upper()} robustness on unseen validation datasets',
]
for dataset_key, specification in DATASETS.items():
    metrics_csv = ARTIFACT_ROOT / dataset_key / TARGET / 'per_sim_metrics.csv'
    if not metrics_csv.is_file():
        raise FileNotFoundError(f'Evaluation did not create {metrics_csv}')
    plot_command.extend([
        '--new_dataset', dataset_key, specification['label'],
        str(specification['conditions']), str(metrics_csv),
    ])
subprocess.run(plot_command, cwd=ROOT, env=SUBPROCESS_ENV, check=True)

## 3. Display the saved figure and statistics

In [ ]:
import pandas as pd
from IPython.display import Image, display

PREFIX = f'fig_unseen_dataset_validation_{TARGET}_delta_rmse'
PNG_PATH = FIGURE_DIR / f'{PREFIX}.png'
SUMMARY_CSV = FIGURE_DIR / f'{PREFIX}_summary.csv'
display(Image(filename=str(PNG_PATH), width=1200))
display(pd.read_csv(SUMMARY_CSV))